# R4 — Short- and long-term memory

The Saffron Shield assistant works with an adjuster across several chats and several days. Within one chat it keeps the whole conversation in the thread, so it never asks the adjuster to repeat themselves. Anything the adjuster asks to save lives outside the chat, in Lakebase Postgres (with the permitted Delta fallback `claims.genai.adjuster_notes`), so a note taken in one chat comes back in a brand-new one. The two scripted sessions from `memory_probes.json` are replayed word for word and every turn is recorded in `evidence/r4_memory.json`.

In [0]:
# Config — the single place the catalog, models, Lakebase and repo paths are named for this notebook
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"

REPO_DIR = "/Workspace/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1/Gen AI - Paper 5 - Scenario Based Assignment"
PROBES_PATH = "/Volumes/claims/raw/landing/genai/memory_probes.json"

# Long-term memory backend: Lakebase Postgres (preferred); Delta fallback if Lakebase is unavailable
LB_ENDPOINT = "projects/claimslakebase/branches/production/endpoints/primary"
LB_DATABASE = "databricks_postgres"
DELTA_NOTES_TABLE = "claims.genai.adjuster_notes"

spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  chat_model={CHAT_MODEL}")

In [0]:
# Lakebase Postgres driver - notebook-scoped install only when the base image lacks it
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("psycopg") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "psycopg[binary,pool]>=3.1.0"])
import psycopg
print("psycopg", psycopg.__version__)

In [0]:
# Long-term memory lives OUTSIDE the chat: Lakebase Postgres when the workspace allows it,
# otherwise the permitted Delta fallback claims.genai.adjuster_notes. Every note the agent
# saves goes through note_save / note_recall below, whichever backend was picked.
import psycopg
from databricks.sdk import WorkspaceClient

_w = WorkspaceClient()
_lb_ep = _w.postgres.get_endpoint(name=LB_ENDPOINT)
_lb_user = _w.current_user.me().user_name

def _q(s: str) -> str:
    return str(s).replace("'", "''")

def _lb_conn():
    # Short-lived OAuth credential minted per connection; never printed or stored.
    token = _w.postgres.generate_database_credential(endpoint=LB_ENDPOINT).token
    return psycopg.connect(host=_lb_ep.status.hosts.host, dbname=LB_DATABASE,
                           user=_lb_user, password=token, sslmode="require", connect_timeout=15)

BACKEND = None
try:
    if _lb_ep.status.disabled:
        raise RuntimeError("Lakebase endpoint is disabled")
    with _lb_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("""CREATE TABLE IF NOT EXISTS adjuster_notes (
                user_id TEXT NOT NULL,
                thread_id TEXT,
                claim_id TEXT NOT NULL,
                note TEXT NOT NULL,
                created_at TIMESTAMPTZ NOT NULL DEFAULT now())""")
        conn.commit()
    BACKEND = "lakebase"
except Exception as e:
    print(f"lakebase unavailable ({str(e)[:160]}) -> falling back to {DELTA_NOTES_TABLE}")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {DELTA_NOTES_TABLE} (
        user_id STRING, thread_id STRING, claim_id STRING, note STRING, created_at TIMESTAMP)""")
    BACKEND = "delta"

def note_save(user_id: str, thread_id: str, claim_id: str, note: str):
    if BACKEND == "lakebase":
        with _lb_conn() as conn:
            with conn.cursor() as cur:
                cur.execute(
                    "INSERT INTO adjuster_notes (user_id, thread_id, claim_id, note) VALUES (%s, %s, %s, %s)",
                    (user_id, thread_id, claim_id, note))
            conn.commit()
    else:
        spark.sql(f"INSERT INTO {DELTA_NOTES_TABLE} "
                  f"SELECT '{_q(user_id)}', '{_q(thread_id)}', '{_q(claim_id)}', '{_q(note)}', current_timestamp()")

def note_recall(user_id: str, claim_id: str):
    if BACKEND == "lakebase":
        with _lb_conn() as conn:
            with conn.cursor() as cur:
                cur.execute(
                    "SELECT claim_id, note, created_at FROM adjuster_notes "
                    "WHERE user_id = %s AND claim_id = %s ORDER BY created_at",
                    (user_id, claim_id))
                rows = cur.fetchall()
    else:
        rows = [(r.claim_id, r.note, str(r.created_at)) for r in spark.sql(
            f"SELECT claim_id, note, created_at FROM {DELTA_NOTES_TABLE} "
            f"WHERE user_id = '{_q(user_id)}' AND claim_id = '{_q(claim_id)}' ORDER BY created_at").collect()]
    return [{"claim_id": r[0], "note": r[1], "saved_at": str(r[2])} for r in rows]

print(f"memory backend: {BACKEND}  (user {_lb_user})")

In [0]:
# The R2 agent (four UC tools, temperature 0) plus two memory tools, with short-term memory
# held in THREADS (the conversation each thread_id keeps) and long-term memory in the backend
# above. save_note / recall_notes are the only tools that cross the chat boundary.
import json
import os

import mlflow
import requests

TOOLS = [
  {"type": "function", "function": {
      "name": "lookup_claim",
      "description": "Look up the stored facts of one claim: status, claim type, admission and discharge dates, billed and approved amounts, rejection code, plan at admission, room rent per day and cap, policy tenure, days since policy start, network flag. Returns facts exactly as stored; does not decide a payout.",
      "parameters": {"type": "object", "properties": {"claim_id": {"type": "string", "description": "Claim id, e.g. CLM-10042"}}, "required": ["claim_id"]}}},
  {"type": "function", "function": {
      "name": "plan_at",
      "description": "Return the plan terms (plan code, family, room rent cap, co-pay pct, PED and initial waiting periods, validity dates) a member held on a given date - the policy version valid on that date, never the member's current plan.",
      "parameters": {"type": "object", "properties": {"member_id": {"type": "string", "description": "Member id, e.g. MEM-0437"}, "as_of_date": {"type": "string", "description": "Date as YYYY-MM-DD"}}, "required": ["member_id", "as_of_date"]}}},
  {"type": "function", "function": {
      "name": "search_policy",
      "description": "Search the Saffron Shield policy binder (hybrid keyword plus vector). Returns at most 5 sections with chunk_id, doc_name, heading and text, best first. Use for any question about what the policy says.",
      "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Natural-language policy question"}}, "required": ["query"]}}},
  {"type": "function", "function": {
      "name": "compute_payable",
      "description": "Deterministically decide a claim's payout by the plan rules. Returns a JSON string with room_rent_over_cap, waiting_period_ok, payable, copay_pct, reason_code and payable_amount. The only source of a payout figure - never compute one yourself.",
      "parameters": {"type": "object", "properties": {
          "plan_code": {"type": "string"},
          "is_pre_existing": {"type": "boolean"},
          "policy_tenure_months": {"type": "integer"},
          "days_since_policy_start": {"type": "integer"},
          "is_network_at_admission": {"type": "boolean"},
          "room_rent_per_day": {"type": "integer"},
          "billed_amount": {"type": "number"}},
        "required": ["plan_code", "is_pre_existing", "policy_tenure_months", "days_since_policy_start", "is_network_at_admission", "room_rent_per_day", "billed_amount"]}}},
  {"type": "function", "function": {
      "name": "save_note",
      "description": "Save a working note for the adjuster, tied to a claim id. Notes persist across chats, so call this whenever the adjuster asks you to save, record, note or remember something. Pass the claim id and the note text exactly as dictated.",
      "parameters": {"type": "object", "properties": {"claim_id": {"type": "string", "description": "Claim id the note belongs to, e.g. CLM-10556"}, "note": {"type": "string", "description": "The note text to keep, exactly as the adjuster dictated it"}}, "required": ["claim_id", "note"]}}},
  {"type": "function", "function": {
      "name": "recall_notes",
      "description": "Return the notes previously saved for a claim id. Notes persist across chats, so use this whenever the adjuster refers to something they noted, saved or recorded earlier - in this chat or a previous one. Returns a JSON list of saved notes.",
      "parameters": {"type": "object", "properties": {"claim_id": {"type": "string", "description": "Claim id, e.g. CLM-10556"}}, "required": ["claim_id"]}}},
]

def run_tool(name: str, args: dict) -> str:
    if name == "lookup_claim":
        rows = spark.sql(f"SELECT * FROM claims.genai.lookup_claim('{_q(args['claim_id'])}')").collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "plan_at":
        rows = spark.sql(f"SELECT * FROM claims.genai.plan_at('{_q(args['member_id'])}', DATE'{_q(args['as_of_date'])}')").collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "search_policy":
        rows = spark.sql(f"SELECT chunk_id, doc_name, heading, text FROM claims.genai.search_policy('{_q(args['query'])}')").collect()
        result = json.dumps([r.asDict() for r in rows], default=str)
    elif name == "compute_payable":
        row = spark.sql(f"""
          SELECT claims.genai.compute_payable(
            '{_q(args['plan_code'])}', {args['is_pre_existing']}, {args['policy_tenure_months']},
            {args['days_since_policy_start']}, {args['is_network_at_admission']},
            {args['room_rent_per_day']}, {args['billed_amount']}) AS payout
        """).collect()[0]
        result = row.payout
    elif name == "save_note":
        note_save(ADJUSTER_USER_ID, CURRENT_THREAD_ID, args["claim_id"], args["note"])
        result = json.dumps({"saved": True, "claim_id": args["claim_id"]})
    elif name == "recall_notes":
        result = json.dumps(note_recall(ADJUSTER_USER_ID, args["claim_id"]), default=str)
    else:
        result = json.dumps({"error": f"unknown tool {name}"})
    return result

w = WorkspaceClient()
AUTH = w.config.authenticate()["Authorization"]
HOST = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

def chat(messages):
    body = {"messages": messages, "tools": TOOLS, "temperature": 0, "max_tokens": 1024}
    r = requests.post(
        f"{HOST}/serving-endpoints/{CHAT_MODEL}/invocations",
        headers={"Authorization": AUTH, "Content-Type": "application/json"},
        json=body,
    )
    r.raise_for_status()
    return r.json()

SYSTEM_PROMPT_MEMORY = """You are the Saffron Shield claims assistant for insurance adjusters, who work on a claim over several days and in several separate chats.
Within the current chat you hold the whole conversation so far: never ask the adjuster to repeat anything they already told you here; use what was said earlier in this chat.
The adjuster's notes live outside the chat. When the adjuster asks you to save, record, note or remember something, call save_note with the claim id and the note text exactly as dictated.
When the adjuster asks about anything they noted, saved or recorded earlier - in this chat or a previous one - call recall_notes with the claim id, then answer from the notes it returns; notes persist across chats.
For a question about the facts, status or amounts of a claim, call the data tools: lookup_claim for the claim's stored facts, and plan_at when plan terms are needed. Always state those facts exactly as the tools return them - never from memory.
For a question about what the policy says (waiting periods, room rent caps, co-pay, process rules), call search_policy, then answer from the returned sections and cite the policy section by its chunk id.
Never invent a payout figure: a payout comes only from the compute_payable tool, checked by code. If a question is not about Saffron Shield claims or policies, say so and answer nothing else."""

os.makedirs(os.path.join(REPO_DIR, "agent", "prompts"), exist_ok=True)
with open(os.path.join(REPO_DIR, "agent", "prompts", "system_memory.md"), "w") as f:
    f.write(SYSTEM_PROMPT_MEMORY)

THREADS = {}
ADJUSTER_USER_ID = None
CURRENT_THREAD_ID = None

@mlflow.trace(name="saffron_shield_memory_agent", span_type="AGENT")
def memory_agent_turn(thread_id: str, user_id: str, question: str, max_steps: int = 4):
    global ADJUSTER_USER_ID, CURRENT_THREAD_ID
    ADJUSTER_USER_ID, CURRENT_THREAD_ID = user_id, thread_id
    history = THREADS.setdefault(thread_id, [])
    messages_before = len(history)
    messages = [{"role": "system", "content": SYSTEM_PROMPT_MEMORY}] + history + [{"role": "user", "content": question}]
    tool_calls_made = []
    for _ in range(max_steps):
        msg = chat(messages)["choices"][0]["message"]
        tcs = msg.get("tool_calls") or []
        if not tcs:
            answer = msg.get("content") or ""
            history.append({"role": "user", "content": question})
            history.append({"role": "assistant", "content": answer})
            mlflow.get_current_active_span().set_attributes({
                "thread_id": thread_id,
                "user_id": user_id,
                "question": question,
                "thread_messages_before": messages_before,
                "tool_calls": json.dumps(tool_calls_made),
                "final_answer": answer,
            })
            return {"answer": answer, "tool_calls": tool_calls_made,
                    "thread_messages_before": messages_before}
        messages.append(msg)
        history.append(msg)
        for tc in tcs:
            name = tc["function"]["name"]
            args = json.loads(tc["function"]["arguments"] or "{}")
            result = run_tool(name, args)
            tool_calls_made.append(name)
            tool_msg = {"role": "tool", "tool_call_id": tc["id"], "content": result}
            messages.append(tool_msg)
            history.append(tool_msg)
    raise RuntimeError("agent did not settle within max_steps")

print(f"memory agent ready ({len(TOOLS)} tools, history kept per thread)")

In [0]:
# Replay the client's two scripted sessions word for word: session 1 (one thread) then
# session 2 (a brand-new thread). One object per turn; sessions kept in probe order.
import json
import os

probes = json.load(open(PROBES_PATH))
sessions_out = []
for s in probes["sessions"]:
    tid = s["thread_id"]
    THREADS.pop(tid, None)  # replay starts each scripted thread from scratch
    turns_out = []
    for i, question in enumerate(s["turns"], 1):
        out = memory_agent_turn(tid, probes["user_id"], question)
        entry = {
            "turn": i,
            "question": question,
            "answer": out["answer"],
            "tool_calls": out["tool_calls"],
            "thread_messages_before": out["thread_messages_before"],
        }
        turns_out.append(entry)
        print(f"s{s['session']} t{i}  before={entry['thread_messages_before']}  tools={entry['tool_calls']}")
        print(f"       {entry['answer'][:130]}")
    sessions_out.append({"session": s["session"], "thread_id": tid, "turns": turns_out})

# Every check in memory_probes.json must hold on what was just recorded.
by_key = {(s["session"], t["turn"]): t for s in sessions_out for t in s["turns"]}
for c in probes["checks"]:
    t = by_key[(c["session"], c["turn"])]
    if c["kind"] == "short_term":
        assert c["must_contain"] in t["answer"], f"short_term s{c['session']} t{c['turn']} missing {c['must_contain']}"
    else:
        assert c["tool"] in t["tool_calls"], f"long_term s{c['session']} t{c['turn']} did not call {c['tool']}"
        assert t["thread_messages_before"] == 0, f"long_term turn ran on a non-empty thread"
        assert c["must_contain"] in t["answer"], f"long_term answer missing {c['must_contain']}"
    print(f"check {c['kind']} s{c['session']} t{c['turn']}: contains '{c['must_contain']}' OK")

assert sessions_out[0]["session"] == 1 and sessions_out[1]["session"] == 2, "sessions must be in order 1, 2"
assert sessions_out[0]["thread_id"] != sessions_out[1]["thread_id"], "the two sessions must use different threads"

EVIDENCE = {
    "backend": BACKEND,
    "user_id": probes["user_id"],
    "sessions": sessions_out,
}
evidence_path = os.path.join(REPO_DIR, "evidence", "r4_memory.json")
os.makedirs(os.path.dirname(evidence_path), exist_ok=True)
with open(evidence_path, "w") as f:
    json.dump(EVIDENCE, f, indent=2)
with open(evidence_path) as f:
    assert json.load(f) == EVIDENCE, "evidence round-trip failed"
print(f"\nwrote {evidence_path}  (backend={BACKEND})")

In [0]:
# Grader-style verification: the evidence file must carry every required key, the checks from
# memory_probes.json must hold on the recorded turns, and the saved note must still be in the
# backend database (outside any chat) after both sessions ended.
import json
import os

ev = json.load(open(os.path.join(REPO_DIR, "evidence", "r4_memory.json")))
probes = json.load(open(PROBES_PATH))

assert set(ev) == {"backend", "user_id", "sessions"}, f"unexpected evidence keys: {sorted(ev)}"
assert ev["backend"] in ("lakebase", "delta"), ev["backend"]
assert ev["user_id"] == probes["user_id"]
assert len(ev["sessions"]) == 2
s1, s2 = ev["sessions"]
assert (s1["session"], s2["session"]) == (1, 2), "sessions must be in order 1, 2"
assert s1["thread_id"] == "memory-session-1" and s2["thread_id"] == "memory-session-2"
assert s1["thread_id"] != s2["thread_id"]

for s in ev["sessions"]:
    for i, t in enumerate(s["turns"], 1):
        assert t["turn"] == i, f"turn numbering broken in session {s['session']}"
        assert t["question"] in probes["sessions"][s["session"] - 1]["turns"], f"s{s['session']} t{i} not word for word"
        assert isinstance(t["tool_calls"], list)
        assert t["answer"].strip(), f"s{s['session']} t{i} empty answer"
        if i == 1:
            assert t["thread_messages_before"] == 0, \
                f"s{s['session']} t1 thread_messages_before={t['thread_messages_before']}, expected 0 on a first turn"
        else:
            # a turn that calls tools leaves the assistant tool-call and tool-result messages
            # in the thread too, so the count grows by more than 2 per turn
            assert t["thread_messages_before"] > s["turns"][i - 2]["thread_messages_before"], \
                f"s{s['session']} t{i} thread_messages_before={t['thread_messages_before']} did not grow with the conversation"

for c in probes["checks"]:
    s = ev["sessions"][c["session"] - 1]
    t = s["turns"][c["turn"] - 1]
    assert c["must_contain"] in t["answer"], f"check {c['kind']} missing {c['must_contain']}"
    if c["kind"] == "long_term":
        assert c["tool"] in t["tool_calls"], f"long_term turn did not call {c['tool']}"
        assert t["thread_messages_before"] == 0, "long_term turn ran on a non-empty thread"
    print(f"{c['kind']:10} s{c['session']} t{c['turn']}: '{c['must_contain']}' in answer OK")

# The note must be in the database itself, independent of any conversation.
if ev["backend"] == "lakebase":
    with _lb_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT note FROM adjuster_notes WHERE user_id = %s AND claim_id = 'CLM-10556'",
                        (ev["user_id"],))
            notes = [r[0] for r in cur.fetchall()]
else:
    notes = [r.note for r in spark.sql(
        f"SELECT note FROM {DELTA_NOTES_TABLE} WHERE user_id = '{_q(ev['user_id'])}' AND claim_id = 'CLM-10556'").collect()]
assert any("ADJ-2219" in n for n in notes), f"saved note not found in backend: {notes}"
print(f"backend {ev['backend']} holds the saved note for CLM-10556 ({len(notes)} note(s))")

print("\nPASS: r4_memory.json records both sessions; note from session 1 returned in session 2")